In [1]:
import java.util.UUID

fun generateId() = "_${UUID.randomUUID().toString().replace("-", "")}"

In [2]:
import kotlin.random.Random

// using aws regions for fun :)
val regions = listOf(
  "us-east-1",
  "us-east-2",
  "us-west-1",
  "us-west-2",
  "af-south-1",
  "ap-east-1",
  "ap-south-2",
  "ap-southeast-3",
  "ap-southeast-5",
  "ap-southeast-4",
  "ap-south-1",
  "ap-northeast-3",
  "ap-northeast-2",
  "ap-southeast-1",
  "ap-southeast-2",
  "ap-east-2",
  "ap-southeast-7",
  "ap-northeast-1",
  "ca-central-1",
  "ca-west-1",
  "eu-central-1",
  "eu-west-1",
  "eu-west-2",
  "eu-south-1",
  "eu-west-3",
  "eu-south-2",
  "eu-north-1",
  "eu-central-2",
  "il-central-1",
  "mx-central-1",
  "me-south-1",
  "me-central-1",
  "sa-east-1",
)

fun generateRandomRegionNames(n: Int): List<String> {
  require(n > 0) { "Number of regions must be greater than 0" }
  require(n <= regions.size) { "Number of regions requested exceeds available regions" }
  return regions.shuffled(Random(System.currentTimeMillis())).take(n)
}

In [3]:
import java.io.File

fun generateProject(
  workspaceDir: String,
  projectName: String
) {
  val projectFile = File("$workspaceDir/$projectName")
  projectFile.mkdirs()

  File("$workspaceDir/$projectName/.project").writeText(
    """
    <?xml version="1.0" encoding="UTF-8"?>
    <projectDescription>
        <name>$projectName</name>
        <comment></comment>
        <projects>
        </projects>
        <buildSpec>
        </buildSpec>
        <natures>
            <nature>org.eclipse.sirius.nature.modelingproject</nature>
        </natures>
    </projectDescription>
    """.trimIndent()
  )
}

In [4]:
@file:DependsOn("org.redundent:kotlin-xml-builder:1.9.3")

import org.redundent.kotlin.xml.xml
import java.io.File

fun generateRingTopology(
  bidirectional: Boolean,
  maxBlockSize: Int,
  meanBlockTime: Double,
  meanTransactionCreationInterval: Double,
  numOfRequiredSecurityConfirmations: Int,
  blockReward: Double,
  numberOfNodes: Int,
  numberOfRegions: Int,
  numberOfNodeAllocations: Int,
  resourcePowers: List<Int>,
  numberOfLinkAllocations: Int,
  linkLatencies: List<List<Pair<Double, Long>>>,
  linkThroughputs: List<List<Pair<Double, Long>>>,
  linkCharacteristicDurations: List<Long>,
  numberOfTransactionTypes: Int,
  transactionSizes: List<Int>,
  transactionAmounts: List<Double>,
  transactionFees: List<Double>,
  blockValidationDurations: List<List<Pair<Double, Int>>>,
  workspaceDir: String,
  projectName: String,
  modelName: String
) {
  generateProject(workspaceDir, projectName)

  val baseId = generateId()
  val networkId = generateId()
  val topologyId = generateId()

  val regions = generateRandomRegionNames(numberOfRegions).map { Pair(generateId(), it) }

  val nodeIds = (0 until numberOfNodes).map { generateId() }
  val linkIds = (0 until numberOfNodes).map { generateId() }

  // Component Repository File
  val blockValidators = blockValidationDurations.map { Pair(generateId(), it) }
  val miningProcessId = generateId()

  val componentRepo = xml("blockchainsystemComponentRepository:BlockchainSystemNodeComponentRepository") {
    attributes(
      "xmi:version" to "2.0",
      "xmlns:xmi" to "http://www.omg.org/XMI",
      "xmlns:xsi" to "http://www.w3.org/2001/XMLSchema-instance",
      "xmlns:blockchainsystemComponentRepository" to "http://palladiosimulator.org/BlockchainSystemComponentModel/BlockchainSystemComponentRepository/1.0",
      "id" to generateId()
    )

    blockValidators.forEach { (blockValidatorId, blockValidationDurations) ->
      "Components" {
        attributes(
          "xsi:type" to "blockchainsystemComponentRepository:BlockValidatorComponent",
          "id" to blockValidatorId
        )
        "ValidationDuration" {
          attribute("id", generateId())

          blockValidationDurations.forEach { (probability, duration) ->
            "Values" {
              attributes(
                "id" to generateId(),
                "Probability" to probability,
                "Duration" to duration
              )
            }
          }
        }
      }
    }

    "Components" {
      attributes(
        "xsi:type" to "blockchainsystemComponentRepository:MiningProcessComponent",
        "id" to miningProcessId,
        "IsMiningProcessEnabled" to "true",
      )
    }
  }
  File("$workspaceDir/$projectName/$modelName.blockchainsystemcomponentrepository").writeText(componentRepo.toString())


  // Node Allocation File
  val nodeAllocations = (0 until numberOfNodeAllocations).map { generateId() }

  val nodeAllocationRepo = xml("nodeallocation:NodeAllocationRepository") {
    attributes(
      "xmi:version" to "2.0",
      "xmlns:xmi" to "http://www.omg.org/XMI",
      "xmlns:xsi" to "http://www.w3.org/2001/XMLSchema-instance",
      "xmlns:blockchainsystemComponentRepository" to "http://palladiosimulator.org/BlockchainSystemComponentModel/BlockchainSystemComponentRepository/1.0",
      "xmlns:nodeallocation" to "http://palladiosimulator.org/BlockchainSystemComponentModel/NodeAllocation/1.0",
      "id" to generateId()
    )

    nodeAllocations.forEach { nodeAllocationId ->
      val blockValidatorAssembly = generateId()
      val miningProcessAssembly = generateId()
      val resourceContainer = generateId()
      val nodeSystem = generateId()
      val environment = generateId()

      "NodeAllocations" {
        attribute("id", nodeAllocationId)

        "AllocationContexts" {
          attributes(
            "id" to generateId(),
            "AssemblyContext" to blockValidatorAssembly,
            "ResourceContainer" to resourceContainer
          )
        }

        "AllocationContexts" {
          attributes(
            "id" to generateId(),
            "AssemblyContext" to miningProcessAssembly,
            "ResourceContainer" to resourceContainer
          )
        }

        "NodeAllocationEnvironment" {
          attribute("id", environment)
          "ResourceContainers" {
            attributes(
              "id" to resourceContainer,
              "ResourcePower" to resourcePowers.random(),
            )
          }
        }

        "NodeSystem" {
          attributes(
            "id" to nodeSystem,
            "BlockValidatorAssembly" to blockValidatorAssembly,
            "MiningProcessAssembly" to miningProcessAssembly,
          )

          "AssemblyContexts" {
            attribute("id", blockValidatorAssembly)
            "EncapsulatedComponent" {
              val blockValidatorId = blockValidators.random().first
              attributes(
                "xsi:type" to "blockchainsystemComponentRepository:BlockValidatorComponent",
                "href" to "$modelName.blockchainsystemcomponentrepository#$blockValidatorId"
              )
            }
          }

          "AssemblyContexts" {
            attribute("id", miningProcessAssembly)
            "EncapsulatedComponent" {
              attributes(
                "xsi:type" to "blockchainsystemComponentRepository:MiningProcessComponent",
                "href" to "$modelName.blockchainsystemcomponentrepository#$miningProcessId"
              )
            }
          }

          "Behavior" {
            // Threesim only has honest nodes
            attribute("id", generateId())
          }
        }

        "GeographicalRegion" {
          val regionId = regions.random().first
          attribute("href", "$modelName.blockchainsystem#$regionId")
        }
      }
    }
  }
  File("$workspaceDir/$projectName/$modelName.nodeallocation").writeText(nodeAllocationRepo.toString())


  // Link Allocation File
  val linkAllocations = (0 until numberOfLinkAllocations).map { generateId() }

  val linkAllocation = xml("linkallocation:LinkAllocationRepository") {
    attributes(
      "xmi:version" to "2.0",
      "xmlns:xmi" to "http://www.omg.org/XMI",
      "xmlns:linkallocation" to "http://palladiosimulator.org/BlockchainSystemComponentModel/LinkAllocation/1.0",
      "id" to generateId()
    )

    linkAllocations.forEach { linkAllocationId ->
      "LinkAllocations" {
        attribute("id", linkAllocationId)

        "latencySpecification" {
          val latency = linkLatencies.random()

          attributes(
            "id" to generateId(),
            "xsi:type" to "linkallocation:DynamicLinkLatencySpecification",
          )

          latency.forEach {
            "Values" {
              attributes(
                "id" to generateId(),
                "Probability" to it.first,
                "Latency" to it.second,
                "Duration" to linkCharacteristicDurations.random()
              )
            }
          }
        }

        "throughputSpecification" {
          val throughput = linkThroughputs.random()

          attributes(
            "id" to generateId(),
            "xsi:type" to "linkallocation:DynamicLinkThroughputSpecification",
          )

          throughput.forEach {
            "Values" {
              attributes(
                "id" to generateId(),
                "Probability" to it.first,
                "Throughput" to it.second,
                "Duration" to linkCharacteristicDurations.random()
              )
            }
          }
        }
      }
    }
  }
  File("$workspaceDir/$projectName/$modelName.linkallocation").writeText(linkAllocation.toString())


  // Blockchain System File
  val blockchainSystem = xml("blockchainsystem:BlockchainSystem") {
    attributes(
      "xmi:version" to "2.0",
      "xmlns:xmi" to "http://www.omg.org/XMI",
      "xmlns:xsi" to "http://www.w3.org/2001/XMLSchema-instance",
      "xmlns:P2PNetwork" to "http://palladiosimulator.org/BlockchainSystemComponentModel/P2PNetwork/1.0",
      "xmlns:blockchainsystem" to "http://palladiosimulator.org/BlockchainSystemComponentModel/BlockchainSystem/1.0",
      "id" to baseId
    )

    "Network" {
      attribute("id", networkId)
      "Topology" {
        attributes(
          "xsi:type" to "P2PNetwork:ExplicitNetworkTopology",
          "id" to topologyId
        )

        nodeIds.forEach { nodeId ->
          "Nodes" {
            attribute("id", nodeId)
            "Allocation" {
              val nodeAllocationId = nodeAllocations.random()
              attribute("href", "$modelName.nodeallocation#$nodeAllocationId")
            }
          }
        }

        nodeIds.forEachIndexed { index, nodeId ->
          val nextNodeId = nodeIds[(index + 1) % nodeIds.size]
          val linkId = linkIds[index]
          "Links" {

            if (bidirectional) {
              attributes(
                "id" to linkId,
                "xsi:type" to "P2PNetwork:BidirectionalLink",
                "ConnectedNodes" to "$nodeId $nextNodeId",
              )
            } else {
              attributes(
                "id" to linkId,
                "xsi:type" to "P2PNetwork:UnidirectionalLink",
                "FromNode" to nodeId,
                "ToNode" to nextNodeId
              )
            }

            "Allocation" {
              val linkAllocationId = linkAllocations.random()
              attribute("href", "$modelName.linkallocation#$linkAllocationId")
            }
          }
        }
      }
    }

    "Specification" {
      attributes(
        "id" to generateId(),
        "BlockReward" to blockReward,
        "MaxBlockSize" to maxBlockSize,
        "MeanBlockTime" to meanBlockTime,
        "NumOfRequiredSecurityConfirmations" to numOfRequiredSecurityConfirmations,
      )
    }

    "TransactionsSpecification" {
      attributes(
        "id" to generateId(),
        "MeanTransactionCreationInterval" to meanTransactionCreationInterval,
      )

      "TransactionPropertiesSpecification" {
        attribute("id", generateId())

        (0 until numberOfTransactionTypes).forEach {
          "Values" {
            attributes(
              "id" to generateId(),
              "Probability" to 1.0 / numberOfTransactionTypes,
              "Size" to transactionSizes.random(),
              "Amount" to transactionAmounts.random(),
              "Fee" to transactionFees.random()
            )
          }
        }
      }
    }

    "GeographicalRegionsSpecification" {
      attribute("id", generateId())
      regions.forEach { (id, name) ->
        "Regions" {
          attributes(
            "id" to id,
            "RegionName" to name
          )
        }
      }
    }
  }
  File("$workspaceDir/$projectName/$modelName.blockchainsystem").writeText(blockchainSystem.toString())


  // Representations File
  val represenations = xml("viewpoint:DAnalysis") {
    attributes(
      "xmi:version" to "2.0",
      "xmlns:xmi" to "http://www.omg.org/XMI",
      "xmlns:viewpoint" to "http://www.eclipse.org/sirius/1.1.0",
      "uid" to generateId(),
      "version" to "15.4.1.202403191723"
    )

    // Add semantic resources
    val semanticResources = listOf(
      "$modelName.linkallocation",
      "$modelName.blockchainsystem",
      "$modelName.blockchainsystemcomponentrepository",
      "$modelName.nodeallocation",
    )

    semanticResources.forEach { resource ->
      "semanticResources" {
        -resource
      }
    }
  }
  File("$workspaceDir/$projectName/representations.aird").writeText(represenations.toString())
}

In [5]:
generateRingTopology(
  bidirectional = true,
  maxBlockSize = 1024, // 1MB block size (in KB)
  meanBlockTime = 60000.0, // 10 minutes average block time (Bitcoin-like)
  meanTransactionCreationInterval = 30.0, // New transaction every 30 milliseconds
  numOfRequiredSecurityConfirmations = 6, // Standard Bitcoin confirmations
  blockReward = 6.25, // Bitcoin block reward
  numberOfNodes = 8, // 8 nodes in the ring
  numberOfRegions = 3,
  numberOfNodeAllocations = 8,
  resourcePowers = listOf(100, 150, 200, 120, 180, 90, 160, 110),
  numberOfLinkAllocations = 2,
  linkLatencies = listOf(
    listOf(
      Pair(0.5, 100), // in bytes per second
      Pair(0.2, 40),
      Pair(0.3, 60),
    ),
    listOf(
      Pair(0.4, 80),
      Pair(0.4, 50),
      Pair(0.2, 30)
    )
  ),
  linkThroughputs = listOf(
    listOf(
      Pair(0.5, 1000), // 1000 transactions per second
      Pair(0.3, 500), // 500 transactions per second
      Pair(0.2, 200) // 200 transactions per second
    ),
    listOf(
      Pair(0.4, 800), // 800 transactions per second
      Pair(0.4, 600), // 600 transactions per second
      Pair(0.2, 300) // 300 transactions per second
    )
  ),
  linkCharacteristicDurations = listOf(1000L, 2000L, 3000L), // Link characteristic durations in milliseconds
  numberOfTransactionTypes = 5,
  transactionSizes = listOf(250, 500, 1000), // Transaction sizes in bytes
  transactionAmounts = listOf(0.1, 1.0, 10.0), // Transaction amounts in currency units
  transactionFees = listOf(0.001, 0.005, 0.01), // Transaction fees
  blockValidationDurations = listOf(
    listOf(
      Pair(0.5, 10),  // 50% chance of 10 seconds validation
      Pair(0.3, 20),  // 30% chance of 20 seconds validation
      Pair(0.2, 30)   // 20% chance of 30 seconds validation
    )
  ),
  workspaceDir = "../threesim-workspace",
  projectName = "threesim-bidirectional-ring",
  modelName = "BidirectionalRing"
)